# Notebook 3: Create Diagnosis table
Creates 20 diagnoses and links them to valid patients/providers with full provider coverage.

In [ ]:
import random
from datetime import date, timedelta
from pyspark.sql import functions as F

SEED = 20260403
random.seed(SEED)

patients = [r.patientId for r in spark.table('patient').select('patientId').collect()]
providers = [r.providerId for r in spark.table('provider').select('providerId').collect()]

if len(patients) != 50:
    raise ValueError('Expected 50 patients from Notebook 1.')
if len(providers) != 20:
    raise ValueError('Expected 20 providers from Notebook 1.')

In [ ]:
# With 20 diagnoses and strict requirement that all 20 providers issue diagnoses,
# each provider appears exactly once in diagnosis rows.
provider_assignments = providers.copy()
random.shuffle(provider_assignments)

icd_catalog = [
    ('I10', 'Essential primary hypertension'),
    ('E11.9', 'Type 2 diabetes mellitus without complications'),
    ('J45.909', 'Unspecified asthma uncomplicated'),
    ('M54.50', 'Low back pain unspecified'),
    ('E78.5', 'Hyperlipidemia unspecified'),
    ('F41.1', 'Generalized anxiety disorder'),
    ('K21.9', 'Gastro-esophageal reflux disease without esophagitis'),
    ('G43.909', 'Migraine unspecified not intractable'),
    ('L30.9', 'Dermatitis unspecified'),
    ('N39.0', 'Urinary tract infection site not specified'),
    ('J06.9', 'Acute upper respiratory infection unspecified'),
    ('R51.9', 'Headache unspecified'),
    ('M25.561', 'Pain in right knee'),
    ('M25.562', 'Pain in left knee'),
    ('E03.9', 'Hypothyroidism unspecified'),
    ('I25.10', 'Atherosclerotic heart disease of native coronary artery'),
    ('F32.A', 'Depression unspecified'),
    ('J30.9', 'Allergic rhinitis unspecified'),
    ('E66.9', 'Obesity unspecified'),
    ('M79.1', 'Myalgia')
]
severities = ['low', 'moderate', 'high']
start_date = date.today() - timedelta(days=90)

diagnoses = []
for i in range(20):
    icd, desc = icd_catalog[i]
    diagnosed = start_date + timedelta(days=random.randint(0, 89))
    diagnoses.append({
        'diagnosisId': f"DX{i+1:03d}",
        'patientId': random.choice(patients),
        'providerId': provider_assignments[i],
        'icdCode': icd,
        'description': desc,
        'severity': random.choice(severities),
        'diagnosedDate': diagnosed.isoformat()
    })

diagnosis_df = spark.createDataFrame(diagnoses)
diagnosis_df = diagnosis_df.select(
    'diagnosisId', 'patientId', 'providerId', 'icdCode', 'description', 'severity', 'diagnosedDate'
)
diagnosis_df = diagnosis_df.withColumn('diagnosedDate', F.to_date('diagnosedDate'))
diagnosis_df.write.mode('overwrite').format('delta').saveAsTable('diagnosis')

display(spark.table('diagnosis').orderBy('diagnosisId'))

In [ ]:
diag = spark.table('diagnosis')

provider_coverage = diag.select('providerId').distinct().count()
patient_fk_orphans = diag.alias('d').join(spark.table('patient').alias('p'), on='patientId', how='left_anti').count()
provider_fk_orphans = diag.alias('d').join(spark.table('provider').alias('p'), on='providerId', how='left_anti').count()

print('diagnosis_count_20:', diag.count() == 20)
print('all_20_providers_covered:', provider_coverage == 20)
print('patient_fk_valid:', patient_fk_orphans == 0)
print('provider_fk_valid:', provider_fk_orphans == 0)